# Benign Trace Tokenization

Parses raw benign session traces (`raw_traces/*.zip`) into the `METHOD:/PATH:/ROLE:/PARAMS:`
token format, then slides a window across each sequence to produce the pretraining corpus
(`benign_windows.jsonl`, 340 windows from 53 sequences across 18 apps/CVEs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/thesis'

## Parse raw YAML traces -> canonicalized token sequences

Path canonicalization: numeric IDs -> `{id}`, UUIDs -> `{uuid}`, hex strings -> `{hex}`,
other query values -> `{val}`.

In [ ]:
import zipfile
import yaml
import re
import json
import os
import glob
from pathlib import Path
from collections import defaultdict

BASE = '/content/drive/MyDrive/thesis'
OUT_DIR = f"{BASE}/data/tokenised"
os.makedirs(OUT_DIR, exist_ok=True)

# Custom loader that ignores unknown Python object tags
class SafeLineLoader(yaml.SafeLoader):
    pass

def unknown_constructor(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node)
    elif isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    else:
        return loader.construct_scalar(node)

SafeLineLoader.add_multi_constructor('', unknown_constructor)

def load_yaml_safe(content):
    try:
        return yaml.load(content, Loader=SafeLineLoader)
    except Exception as e:
        return None

def canonicalise_path(url):
    path = re.sub(r'https?://[^/]+', '', url)
    path = re.sub(r'/\d+', '/{id}', path)
    path = re.sub(r'/[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', '/{uuid}', path)
    path = re.sub(r'/[0-9a-f]{16,}', '/{hex}', path)
    path = re.sub(r'=([^&]+)', '={val}', path)
    path = re.sub(r'/+', '/', path)
    return path.strip()

def tokenise_request(yaml_data):
    method = yaml_data.get('method', 'GET')
    url = yaml_data.get('url', '')
    role = yaml_data.get('role', 'Anonymous')
    post_data = yaml_data.get('post_data_json', None)
    canonical_path = canonicalise_path(url)
    token = f"METHOD:{method} PATH:{canonical_path} ROLE:{role}"
    if post_data and isinstance(post_data, dict):
        param_names = sorted(post_data.keys())
        if param_names:
            token += f" PARAMS:{','.join(param_names)}"
    return token

def numeric_sort_key(name):
    stem = Path(name).stem
    try:
        return int(stem)
    except ValueError:
        return float('inf')

# Find all zip files in raw_traces
RAW_DIR = f"{BASE}/data/raw_traces"
all_zips = sorted(glob.glob(f"{RAW_DIR}/*.zip"))
print(f"Found {len(all_zips)} zip files:")
for z in all_zips:
    print(f"  {os.path.basename(z)}")

# Build index across all zips
cve_role_files = defaultdict(lambda: defaultdict(list))

for zip_path in all_zips:
    with zipfile.ZipFile(zip_path, 'r') as z:
        for name in z.namelist():
            parts = Path(name).parts
            if len(parts) == 3 and parts[2].endswith('.yaml') and parts[2] != 'paramval.yaml':
                cve, role, filename = parts
                cve_role_files[cve][role].append((filename, zip_path))

print(f"\nFound {len(cve_role_files)} CVEs total")

# Tokenise
all_sequences = []
stats = {}

for cve_name in sorted(cve_role_files.keys()):
    cve_sequences = []
    cve_total = 0
    skipped = 0

    for role_name in sorted(cve_role_files[cve_name].keys()):
        file_entries = sorted(
            cve_role_files[cve_name][role_name],
            key=lambda x: numeric_sort_key(x[0])
        )

        tokens = []
        for filename, zip_path in file_entries:
            zip_inner = f"{cve_name}/{role_name}/{filename}"
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    with z.open(zip_inner) as f:
                        content = f.read()
                data = load_yaml_safe(content)
                if data and isinstance(data, dict):
                    token = tokenise_request(data)
                    tokens.append(token)
                else:
                    skipped += 1
            except Exception as e:
                skipped += 1

        if tokens:
            sequence = " [SEP] ".join(tokens)
            cve_sequences.append({
                "cve": cve_name,
                "role": role_name,
                "sequence": sequence,
                "length": len(tokens)
            })
            cve_total += len(tokens)

    stats[cve_name] = {
        "sequences": len(cve_sequences),
        "total_requests": cve_total
    }
    all_sequences.extend(cve_sequences)
    print(f"{cve_name}: {len(cve_sequences)} sequences, {cve_total} requests (skipped: {skipped})")

print(f"\nTotal sequences: {len(all_sequences)}")
print(f"Total requests: {sum(s['length'] for s in all_sequences)}")

output_file = f"{OUT_DIR}/benign_sequences.jsonl"
with open(output_file, 'w') as f:
    for seq in all_sequences:
        f.write(json.dumps(seq) + '\n')

stats_file = f"{OUT_DIR}/tokenisation_stats.json"
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\nSaved to {output_file}")

## Quick check

In [ ]:
with open(f"{BASE}/data/tokenised/benign_sequences.jsonl") as f:
    lines = f.readlines()
print(f"Total sequences saved: {len(lines)}")

import json
sample = json.loads(lines[0])
print(f"Sample -- CVE: {sample['cve']} | Role: {sample['role']} | Length: {sample['length']}")
print(f"Preview: {sample['sequence'][:300]}")

## Sliding window -> pretraining corpus (benign_windows.jsonl)

In [ ]:
import json
import os

WINDOW_SIZE = 50    # requests per window
STRIDE = 25          # overlap between windows

OUT_DIR = f"{BASE}/data/tokenised"
input_file = f"{OUT_DIR}/benign_sequences.jsonl"
output_file = f"{OUT_DIR}/benign_windows.jsonl"

windowed = []

with open(input_file) as f:
    sequences = [json.loads(line) for line in f]

for seq in sequences:
    tokens = seq['sequence'].split(' [SEP] ')
    cve = seq['cve']
    role = seq['role']

    for start in range(0, len(tokens), STRIDE):
        end = start + WINDOW_SIZE
        window_tokens = tokens[start:end]

        if len(window_tokens) < 10:
            continue

        windowed.append({
            "cve": cve,
            "role": role,
            "sequence": " [SEP] ".join(window_tokens),
            "length": len(window_tokens),
            "window_start": start
        })

print(f"Original sequences: {len(sequences)}")
print(f"Windowed samples: {len(windowed)}")

with open(output_file, 'w') as f:
    for w in windowed:
        f.write(json.dumps(w) + '\n')

print(f"Saved to {output_file}")